# AlphaForge — Model Analysis
Compare TFT vs LSTM baseline. Backtest tearsheet. SHAP analysis.


In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.config import get_settings

settings = get_settings()
mlflow.set_tracking_uri(settings.mlflow_tracking_uri)

# Load all experiment runs
runs = mlflow.search_runs(
    experiment_names=[settings.mlflow_experiment_name],
    order_by=['metrics.cv_auc_mean DESC'],
)
print(f'Total runs: {len(runs)}')
cols = ['run_id', 'params.model', 'metrics.cv_auc_mean', 'metrics.cv_f1_mean',
        'metrics.backtest/sharpe_ratio', 'metrics.backtest/max_drawdown']
existing = [c for c in cols if c in runs.columns]
runs[existing].head(10)

In [ ]:
# ── Model comparison: TFT vs LSTM ────────────────────────────────
metrics_to_compare = {
    'CV AUC-ROC': 'metrics.cv_auc_mean',
    'CV F1': 'metrics.cv_f1_mean',
    'Sharpe Ratio': 'metrics.backtest/sharpe_ratio',
    'Max Drawdown': 'metrics.backtest/max_drawdown',
}

if 'params.model' in runs.columns:
    summary = runs.groupby('params.model')[list(metrics_to_compare.values())].mean()
    summary.columns = list(metrics_to_compare.keys())
    print('Model Comparison (averaged across runs):')
    print(summary.round(4))
else:
    print('Run models first via: make train && make backtest')

In [ ]:
# ── Backtest equity curve ─────────────────────────────────────────
# Load a saved backtest result and plot the equity curve
from src.data.storage import get_engine
from sqlalchemy import text

with get_engine().connect() as conn:
    results = pd.read_sql(
        text('SELECT * FROM backtest_results ORDER BY created_at DESC LIMIT 5'),
        conn
    )

if not results.empty:
    print('Latest backtest results:')
    display_cols = ['model_version', 'strategy', 'sharpe_ratio', 'sortino_ratio',
                    'max_drawdown', 'win_rate', 'total_trades', 'calmar_ratio']
    existing = [c for c in display_cols if c in results.columns]
    print(results[existing].to_string())
else:
    print('No backtest results yet. Run: make backtest')

In [ ]:
# ── Calibration curve ─────────────────────────────────────────────
# Are the model's predicted probabilities well-calibrated?
from sklearn.calibration import calibration_curve
import numpy as np

# Simulate calibration analysis with stored predictions
from src.data.storage import get_engine
from sqlalchemy import text

with get_engine().connect() as conn:
    preds = pd.read_sql(
        text('SELECT prob_up, signal FROM predictions ORDER BY time DESC LIMIT 5000'),
        conn
    )

if not preds.empty:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(preds['prob_up'], bins=50, color='#00d4ff', alpha=0.7)
    ax.set_xlabel('Predicted P(up)')
    ax.set_ylabel('Count')
    ax.set_title('Signal Probability Distribution')
    ax.axvline(0.5, color='#f59e0b', linestyle='--', label='Decision boundary')
    ax.legend()
    ax.set_facecolor('#0b1020')
    plt.show()
else:
    print('No predictions yet. Run: make serve, then call /predict')

In [ ]:
# ── Optuna hyperparameter search results ─────────────────────────
import optuna

try:
    study = optuna.load_study(
        study_name='alphaforge-lstm-search',
        storage='sqlite:///optuna.db'
    )
    print(f'Best trial: {study.best_trial.number}')
    print(f'Best value (CV AUC): {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')
    
    # Importance plot
    importances = optuna.importance.get_param_importances(study)
    fig, ax = plt.subplots(figsize=(8, 5))
    params = list(importances.keys())
    values = list(importances.values())
    ax.barh(params, values, color='#7c3aed')
    ax.set_title('Hyperparameter Importance')
    ax.set_facecolor('#0b1020')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'No Optuna study found yet. Run: make hparam-search')